# Workshop Data Setup: Northgate Advisory Sample Data

The facilitator runs this notebook once before the workshop. It copies the ready-made sample data from the `data/` folder next to this notebook into the workshop schema and volume, so every workshop starts from exactly the same data. It takes a few seconds.

| What | Where it ends up |
| --- | --- |
| `employees`, `clients`, `engagements`, `engagement_assignments` | Delta tables in the workshop schema |
| Timesheet CSVs, expense JSON, rate card and FX CSVs | `timesheets/`, `expenses/` and `reference/` in the volume |
| 16 Statement of Work PDFs and 1 change request | `contracts/` in the volume |
| Answer key for the AI extraction exercise, and the pre-built Designer prep `sow_extraction_demo.designer.ipynb` for [3. Extension - PDF Parsing](#notebook-102205435070834) | `_facilitator/` in the volume (Designer exercises never read it) |

**How to use it**
1. Set the `catalog`, `schema` and `volume` parameters. They must match the parameters in the tutorial notebooks.
2. Run the next three cells in order. They create the schema and volume if needed, copy the raw files and load the four tables. Each file is checked against the checksums in `data/MANIFEST.json`, so you know the copy is exact. Running it again resets the data, including removing the November file.
3. During Part 3 of [2. Lakeflow Designer](#notebook-3452153026155251) (job scheduling), run **Drop November Timesheet File** at the end.

To change the scenario itself, see [Maintainer - Regenerate Sample Data](#notebook-3452153026155252). Facilitators don't need it.

## Load the workshop data

Run the next three cells in order. They create the schema and volume if needed, copy the raw files and load the four tables.

In [0]:
import hashlib, json, os, shutil
from datetime import date

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")

FQ_SCHEMA = f"`{CATALOG}`.`{SCHEMA}`"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATA_DIR = os.path.join(os.getcwd(), "data")  # the shipped sample data, next to this notebook
MANIFEST = json.load(open(f"{DATA_DIR}/MANIFEST.json"))

def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ_SCHEMA} COMMENT 'Lakeflow Designer workshop: synthetic consulting billing and utilisation data'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQ_SCHEMA}.`{VOLUME}` COMMENT 'Raw source files: timesheet exports, expense claims, reference CSVs and Statement of Work PDFs'")

print(f"Schema : {CATALOG}.{SCHEMA}")
print(f"Volume : {VOLUME_PATH}")
print(f"Source : {DATA_DIR} ({len(MANIFEST['files'])} files, {len(MANIFEST['tables'])} tables)")

In [0]:
FOLDERS = ["timesheets", "expenses", "reference", "contracts", "_facilitator"]

# Remove timesheet files that aren't in the shipped data (for example the November file from an earlier run)
import glob
for old_file in glob.glob(f"{VOLUME_PATH}/timesheets/*.csv"):
    if f"timesheets/{os.path.basename(old_file)}" not in MANIFEST["files"]:
        os.remove(old_file)
        print(f"Removed {old_file}")

copied = {}
for rel, checksum in MANIFEST["files"].items():
    folder = rel.split("/")[0]
    if folder not in FOLDERS:
        continue  # tables/ are loaded below; new_month/ is used in Part 3
    os.makedirs(f"{VOLUME_PATH}/{folder}", exist_ok=True)
    shutil.copyfile(f"{DATA_DIR}/{rel}", f"{VOLUME_PATH}/{rel}")
    if md5(f"{VOLUME_PATH}/{rel}") != checksum:
        raise RuntimeError(f"{rel} doesn't match MANIFEST.json. Has the data/ folder been edited?")
    copied[folder] = copied.get(folder, 0) + 1

# The pre-built Designer prep for the PDF extension demo. It was saved against the original catalog and schema,
# so point its source path, output catalog/schema and table names at this workshop's catalog, schema and volume.
import re
DEMO_PREP = "designer_demo/sow_extraction_demo.designer.ipynb.json"
if md5(f"{DATA_DIR}/{DEMO_PREP}") != MANIFEST["files"][DEMO_PREP]:
    raise RuntimeError(f"{DEMO_PREP} doesn't match MANIFEST.json. Has the data/ folder been edited?")
with open(f"{DATA_DIR}/{DEMO_PREP}") as f:
    demo_prep = f.read()
demo_prep = demo_prep.replace("/Volumes/praju_stable_2_catalog/consulting_billing/raw_files", VOLUME_PATH)
demo_prep = re.sub(r"\bpraju_stable_2_catalog\b", CATALOG, demo_prep)
demo_prep = re.sub(r"\bconsulting_billing\b", SCHEMA, demo_prep)
json.loads(demo_prep)  # still a valid file
with open(f"{VOLUME_PATH}/_facilitator/sow_extraction_demo.designer.ipynb", "w") as f:
    f.write(demo_prep)
copied["_facilitator"] += 1

print(f"Copied {sum(copied.values())} files to {VOLUME_PATH}, and every checksum matches MANIFEST.json:")
for folder in FOLDERS:
    print(f"  {folder + '/':<14} {copied.get(folder, 0):>3} files")

In [0]:
for name, spec in MANIFEST["tables"].items():
    fields = [c.strip().split() for c in spec["ddl"].split(",")]
    cols = [f[0] for f in fields]
    date_cols = {f[0] for f in fields if f[1] == "DATE"}
    with open(f"{DATA_DIR}/tables/{name}.jsonl") as f:
        rows = [tuple(date.fromisoformat(r[c]) if c in date_cols and r[c] else r[c] for c in cols)
                for r in map(json.loads, f)]

    spark.createDataFrame(rows, spec["ddl"]).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ_SCHEMA}.{name}")
    spark.sql(f"COMMENT ON TABLE {FQ_SCHEMA}.{name} IS '{spec['comment']}'")

    n = spark.table(f"{FQ_SCHEMA}.{name}").count()
    if n != spec["rows"]:
        raise RuntimeError(f"{name}: expected {spec['rows']} rows, loaded {n}")
    print(f"{name:<24} {n:>4} rows")

## Simulate a new month (run during Part 3 of the tutorial)

The next cell copies `timesheets_2026_11.csv` (2-6 November 2026) into the timesheets folder, as if the timesheet system had exported a new month. Run it during the scheduling exercise, then re-run the scheduled visual data prep and watch the November rows appear. Re-running the cells above removes it again.

In [0]:
import os, shutil

volume_path = f"/Volumes/{dbutils.widgets.get('catalog')}/{dbutils.widgets.get('schema')}/{dbutils.widgets.get('volume')}"
source = os.path.join(os.getcwd(), "data", "new_month", "timesheets_2026_11.csv")
shutil.copyfile(source, f"{volume_path}/timesheets/timesheets_2026_11.csv")
print(f"Copied timesheets_2026_11.csv to {volume_path}/timesheets")